In [28]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Obtiene la ruta del directorio actual donde está este archivo/notebook
env_path = Path.cwd() / "pass.env"

# Cargar especificando la ruta exacta y sobreescribir variables existentes
loaded = load_dotenv(dotenv_path=env_path, override=True)

# Diagnóstico rápido: load_dotenv devuelve True si encontró y leyó el archivo
print(f"¿Archivo encontrado y cargado?: {loaded}")

PG_CONNECTION_STRING = os.getenv("PG_CONNECTION_STRING")

# Verificación detallada de cuál variable falta
if not PG_CONNECTION_STRING:
    raise ValueError("Falta PG_CONNECTION_STRING en pass.env")

print("Variables cargadas correctamente.")

¿Archivo encontrado y cargado?: True
Variables cargadas correctamente.


In [29]:
import psycopg2

conn = psycopg2.connect(PG_CONNECTION_STRING)
cur = conn.cursor()

# Ejecución "dummy": no toca ninguna tabla nuestra, solo confirma que el motor responde
cur.execute("SELECT version();")
print(" Conectado a PostgreSQL")
print(cur.fetchone()[0])


 Conectado a PostgreSQL
PostgreSQL 18.6 (c5250a2) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [30]:
conn.rollback()

cur.execute("DROP TABLE IF EXISTS factura_detalle CASCADE;")
cur.execute("DROP TABLE IF EXISTS facturas CASCADE;")
cur.execute("DROP TABLE IF EXISTS prescripciones CASCADE;")
cur.execute("DROP TABLE IF EXISTS medicamentos CASCADE;")
cur.execute("DROP TABLE IF EXISTS observaciones CASCADE;")
cur.execute("DROP TABLE IF EXISTS encuentros CASCADE;")
cur.execute("DROP TABLE IF EXISTS reportes_previos CASCADE;")
cur.execute("DROP TABLE IF EXISTS antecedentes CASCADE;")
cur.execute("DROP TABLE IF EXISTS consultas CASCADE;")
cur.execute("DROP TABLE IF EXISTS pacientes CASCADE;")
cur.execute("DROP TABLE IF EXISTS auditoria_cambios CASCADE;")
cur.execute("DROP TABLE IF EXISTS usuarios CASCADE;")
cur.execute("DROP TABLE IF EXISTS roles CASCADE;")

cur.execute("""
CREATE TABLE roles (
    id_rol       SERIAL PRIMARY KEY,
    nombre       VARCHAR(50) NOT NULL UNIQUE
                 CHECK (nombre IN ('Admin', 'Medico', 'Administrativo', 'Paciente')),
    descripcion  TEXT,
    created_at   TIMESTAMP DEFAULT now(),
    is_active    BOOLEAN DEFAULT true
)""");

cur.execute("""
CREATE TABLE usuarios (
    numero_documento_usuario BIGINT PRIMARY KEY,
    id_rol           INTEGER NOT NULL REFERENCES roles(id_rol),

    username         VARCHAR(50) NOT NULL UNIQUE,
    password_hash    TEXT NOT NULL,

    nombres          VARCHAR(100) NOT NULL,
    apellidos        VARCHAR(100) NOT NULL,
    email            VARCHAR(150),
    telefono         VARCHAR(20),

    estado           BOOLEAN DEFAULT true,

    created_at       TIMESTAMP DEFAULT now(),
    updated_at       TIMESTAMP,

    is_deleted       BOOLEAN DEFAULT false,
    deleted_at       TIMESTAMP,
    deleted_by       BIGINT REFERENCES usuarios(numero_documento_usuario)
)""");

cur.execute("""
CREATE TABLE auditoria_cambios (
    id_auditoria BIGSERIAL PRIMARY KEY,

    tabla_afectada VARCHAR(80) NOT NULL,

    registro_id VARCHAR(100) NOT NULL,

    accion VARCHAR(20) NOT NULL
        CHECK (accion IN (
            'CREAR',
            'EDITAR',
            'ELIMINAR',
            'RESTAURAR'
        )),

    datos_anteriores JSONB,

    datos_nuevos JSONB,

    realizado_por BIGINT NOT NULL
        REFERENCES usuarios(numero_documento_usuario),

    fecha_hora TIMESTAMPTZ NOT NULL DEFAULT now()
)""");

cur.execute("""
CREATE TABLE pacientes (
    numero_documento_paciente BIGINT PRIMARY KEY,

    id_usuario        BIGINT UNIQUE REFERENCES usuarios(numero_documento_usuario),

    tipo_documento     VARCHAR(20) NOT NULL,

    nombres             VARCHAR(100) NOT NULL,
    apellidos           VARCHAR(100) NOT NULL,

    fecha_nacimiento    DATE,
    sexo                VARCHAR(20),

    telefono            VARCHAR(20),
    direccion           VARCHAR(200),

    municipio_residencia VARCHAR(100),
    zona_residencia      VARCHAR(20) CHECK (zona_residencia IN ('urbana', 'rural_dispersa')),

    created_at          TIMESTAMP DEFAULT now(),
    updated_at          TIMESTAMP,

    is_deleted          BOOLEAN DEFAULT false,
    deleted_at          TIMESTAMP,
    deleted_by          BIGINT REFERENCES usuarios(numero_documento_usuario)
)""");

cur.execute("""
CREATE TABLE antecedentes (
    id_antecedente   SERIAL PRIMARY KEY,

    id_paciente      BIGINT NOT NULL REFERENCES pacientes(numero_documento_paciente),

    tipo             VARCHAR(50) NOT NULL,
    codigo           VARCHAR(50),
    descripcion      TEXT NOT NULL,

    fecha_registro   TIMESTAMP DEFAULT now(),

    registrado_por   BIGINT NOT NULL REFERENCES usuarios(numero_documento_usuario),

    is_deleted       BOOLEAN DEFAULT false,
    deleted_at       TIMESTAMP,
    deleted_by       BIGINT REFERENCES usuarios(numero_documento_usuario))""");

cur.execute("""
CREATE TABLE reportes_previos (
    id_reporte             SERIAL PRIMARY KEY,

    id_paciente            BIGINT NOT NULL REFERENCES pacientes(numero_documento_paciente),

    fecha_hora_reporte     TIMESTAMP NOT NULL,

    sintoma_principal      TEXT NOT NULL,
    inicio_sintomas        TIMESTAMP,
    evolucion              TEXT,

    signos_alarma_presentes BOOLEAN DEFAULT false,
    descripcion_signos_alarma TEXT,

    ubicacion_aproximada   VARCHAR(200),
    municipio_origen       VARCHAR(100),

    distancia_aproximada_km DECIMAL(10,2),
    tiempo_desplazamiento_min INTEGER,

    orientacion_inicial    TEXT,

    registrado_por         BIGINT NOT NULL REFERENCES usuarios(numero_documento_usuario),

    created_at             TIMESTAMP DEFAULT now(),
    updated_at              TIMESTAMP,

    is_deleted              BOOLEAN DEFAULT false,
    deleted_at              TIMESTAMP,
    deleted_by              BIGINT REFERENCES usuarios(numero_documento_usuario))""");

cur.execute("""
CREATE TABLE encuentros (
    id_encuentro       SERIAL PRIMARY KEY,

    id_paciente        BIGINT NOT NULL REFERENCES pacientes(numero_documento_paciente),

    fecha_hora_ingreso TIMESTAMP NOT NULL,
    fecha_hora_fin     TIMESTAMP,

    tipo_encuentro     VARCHAR(50),
    servicio           VARCHAR(50) DEFAULT 'URGENCIAS',
    estado             VARCHAR(30) CHECK (estado IN ('en_triage', 'en_atencion', 'en_observacion', 'finalizado')),

    motivo_consulta         TEXT,
    observaciones_generales TEXT,

    -- Información de triage integrada al encuentro
    nivel_triage SMALLINT CHECK (nivel_triage BETWEEN 1 AND 5),
    fecha_hora_triage  TIMESTAMP,
    dolor_escala       INTEGER CHECK (dolor_escala BETWEEN 0 AND 10),
    observaciones_triage TEXT,

    clasificado_por    BIGINT REFERENCES usuarios(numero_documento_usuario),
    clasificacion_automatica BOOLEAN DEFAULT false,

    creado_por         BIGINT NOT NULL REFERENCES usuarios(numero_documento_usuario),

    created_at         TIMESTAMP DEFAULT now(),
    updated_at         TIMESTAMP,

    is_deleted         BOOLEAN DEFAULT false,
    deleted_at         TIMESTAMP,
    deleted_by         BIGINT REFERENCES usuarios(numero_documento_usuario)
)""");

cur.execute("""
CREATE TABLE observaciones (
    id_observacion     SERIAL PRIMARY KEY,

    id_encuentro       INTEGER NOT NULL REFERENCES encuentros(id_encuentro),

    tipo_observacion   VARCHAR(100) NOT NULL,

    codigo_loinc       VARCHAR(30),

    nombre             VARCHAR(150) NOT NULL,

    valor_numerico     DECIMAL(10,2),
    valor_texto        TEXT,
    unidad             VARCHAR(30),

    fecha_hora_observacion TIMESTAMP NOT NULL,

    registrado_por     BIGINT NOT NULL REFERENCES usuarios(numero_documento_usuario),

    created_at         TIMESTAMP DEFAULT now(),
    updated_at         TIMESTAMP,

    is_deleted         BOOLEAN DEFAULT false,
    deleted_at         TIMESTAMP,
    deleted_by         BIGINT REFERENCES usuarios(numero_documento_usuario)
)""");

cur.execute("""
CREATE TABLE medicamentos (
    codigo_cum         VARCHAR(50) PRIMARY KEY,

    nombre             VARCHAR(150) NOT NULL,
    principio_activo   VARCHAR(150),
    concentracion      VARCHAR(100),
    forma_farmaceutica VARCHAR(100),

    registro_sanitario VARCHAR(100),
    estado_cum         VARCHAR(30),

    precio_unitario    DECIMAL(12,2)
)""");

cur.execute("""
CREATE TABLE prescripciones (
    id_prescripcion    SERIAL PRIMARY KEY,

    id_encuentro       INTEGER NOT NULL REFERENCES encuentros(id_encuentro),
    codigo_cum         VARCHAR(50) NOT NULL REFERENCES medicamentos(codigo_cum),

    dosis              VARCHAR(100),
    frecuencia         VARCHAR(100),
    via_administracion VARCHAR(50),
    cantidad           INTEGER NOT NULL,

    prescrito_por      BIGINT NOT NULL REFERENCES usuarios(numero_documento_usuario),
    fecha_prescripcion TIMESTAMP NOT NULL DEFAULT now(),

    estado             VARCHAR(30) DEFAULT 'activa'
                        CHECK (estado IN ('activa', 'dispensada', 'anulada')),

    created_at         TIMESTAMP DEFAULT now(),
    updated_at         TIMESTAMP,

    is_deleted         BOOLEAN DEFAULT false,
    deleted_at         TIMESTAMP,
    deleted_by         BIGINT REFERENCES usuarios(numero_documento_usuario)
)""");

cur.execute("""
CREATE TABLE facturas (
    id_factura      SERIAL PRIMARY KEY,

    id_paciente     BIGINT NOT NULL REFERENCES pacientes(numero_documento_paciente),
    id_encuentro    INTEGER NOT NULL REFERENCES encuentros(id_encuentro),

    numero_factura  VARCHAR(50) NOT NULL UNIQUE,

    fecha_emision   TIMESTAMP NOT NULL,

    concepto        TEXT,

    total           DECIMAL(12,2),

    estado          VARCHAR(30) CHECK (estado IN ('pendiente', 'pagada', 'anulada')),

    creado_por      BIGINT NOT NULL REFERENCES usuarios(numero_documento_usuario),

    created_at      TIMESTAMP DEFAULT now(),
    updated_at      TIMESTAMP,

    is_deleted      BOOLEAN DEFAULT false,
    deleted_at      TIMESTAMP,
    deleted_by      BIGINT REFERENCES usuarios(numero_documento_usuario)
)""");

cur.execute("""
CREATE TABLE factura_detalle (
    id_detalle       SERIAL PRIMARY KEY,

    id_factura       INTEGER NOT NULL REFERENCES facturas(id_factura),
    id_prescripcion  INTEGER REFERENCES prescripciones(id_prescripcion),

    concepto         VARCHAR(200) NOT NULL,
    cantidad         INTEGER NOT NULL DEFAULT 1,
    valor_unitario   DECIMAL(12,2) NOT NULL,
    valor_total      DECIMAL(12,2) NOT NULL,

    created_at       TIMESTAMP DEFAULT now(),

    is_deleted       BOOLEAN DEFAULT false,
    deleted_at       TIMESTAMP,
    deleted_by       BIGINT REFERENCES usuarios(numero_documento_usuario)
)""");

conn.commit()
print(" Tablas creadas en PostgreSQL.")



 Tablas creadas en PostgreSQL.
